# Demo: Equifinality in Model Calibration

## Objectives

This notebook demonstrates **equifinality** - a fundamental challenge in model calibration where multiple parameter combinations produce equally good fits to observations. We explore:

- How parameter interactions create non-unique solutions
- Why the objective function surface matters
- The concept of parameter correlation and identifiability
- Implications for uncertainty in calibrated models

## Background

In the previous notebook (`calibration_concept.ipynb`), we calibrated a single parameter (K) and found a unique minimum. Real groundwater models have multiple parameters that often interact, leading to **equifinality**:

> 📚 **Definition: Equifinality**
> 
> Equifinality occurs when different parameter combinations produce model outputs that are equally consistent with observations. This means:
> - The calibration problem has no unique solution
> - Many "equally good" parameter sets exist
> - Parameter uncertainty may be much larger than residuals suggest
>
> First described by Beven & Binley (1992) in the context of hydrological modeling.

In [ ]:
# Import libraries
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
from matplotlib.colors import LogNorm
from mpl_toolkits.mplot3d import Axes3D
from IPython.display import display
import ipywidgets as widgets

# Set up plotting
%matplotlib widget
plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['font.size'] = 10

## 1. The Two-Parameter Model

We use the same 1D groundwater model, but now treat both **K** (hydraulic conductivity) and **R** (recharge rate) as unknown parameters.

The steady-state head distribution is:

h(x) = h₀ + (h₁ - h₀)(x/L) + **(R/K)** × (L²/2) × (x/L) × (1 - x/L)

> ⚠️ **Key Insight: Parameter Ratio**
> 
> Notice that K and R only appear as the **ratio R/K** in the equation!
> 
> This means:
> - K=10, R=0.001 gives the same head as K=20, R=0.002
> - The model cannot distinguish between these combinations
> - We have **structural non-identifiability**

In [ ]:
def simple_gw_model(K, R, L=1000, h0=100, h1=95, nx=50):
    """
    Simple 1D groundwater model with recharge.
    
    Args:
        K: Hydraulic conductivity (m/day)
        R: Recharge rate (m/day)
        L: Domain length (m)
        h0: Head at left boundary (m)
        h1: Head at right boundary (m)
        nx: Number of spatial points
    
    Returns:
        x: Spatial coordinates (m)
        h: Hydraulic head (m)
    """
    x = np.linspace(0, L, nx)
    
    # Linear component (boundary conditions)
    h_linear = h0 + (h1 - h0) * (x / L)
    
    # Parabolic component - depends on R/K ratio
    h_recharge = (R / K) * (L**2 / 2) * (x/L) * (1 - x/L)
    
    h = h_linear + h_recharge
    
    return x, h

## 2. Generate Synthetic Observations

We create observations using "true" parameter values.

In [ ]:
# "True" parameter values (unknown in real problems)
K_true = 15.0    # m/day
R_true = 0.001   # m/day (= 1 mm/day = 365 mm/year)

# Model domain parameters
L = 1000  # Domain length (m)
h0 = 100  # Left boundary head (m)
h1 = 95   # Right boundary head (m)

# Generate "truth"
x_full, h_true = simple_gw_model(K_true, R_true, L, h0, h1)

# Observation well locations
obs_locations = np.array([150, 350, 500, 650, 850])
obs_indices = [np.argmin(np.abs(x_full - loc)) for loc in obs_locations]

# Sample observations with measurement noise
np.random.seed(42)
noise_std = 0.15
h_obs = h_true[obs_indices] + np.random.normal(0, noise_std, len(obs_indices))
x_obs = x_full[obs_indices]

# Calculate the true R/K ratio
ratio_true = R_true / K_true

print(f"True parameters:")
print(f"  K = {K_true} m/day")
print(f"  R = {R_true} m/day ({R_true*1000:.1f} mm/day)")
print(f"  R/K ratio = {ratio_true:.6f}")

## 3. The Objective Function Surface

Let's visualize how the objective function (SSR) varies across the 2D parameter space.

In [ ]:
def objective_function(K, R):
    """
    Calculate Sum of Squared Residuals for given K and R.
    """
    x, h = simple_gw_model(K, R, L, h0, h1)
    h_sim = h[obs_indices]
    residuals = h_sim - h_obs
    return np.sum(residuals**2)


# Create parameter grid
K_range = np.linspace(5, 40, 100)
R_range = np.linspace(0.0002, 0.003, 100)
K_grid, R_grid = np.meshgrid(K_range, R_range)

# Calculate SSR for each parameter combination
SSR_grid = np.zeros_like(K_grid)
for i in range(K_grid.shape[0]):
    for j in range(K_grid.shape[1]):
        SSR_grid[i, j] = objective_function(K_grid[i, j], R_grid[i, j])

In [ ]:
# Plot the objective function surface
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Left: Contour plot
ax1 = axes[0]
levels = np.logspace(np.log10(SSR_grid.min() + 0.001), np.log10(SSR_grid.max()), 30)
contour = ax1.contourf(K_grid, R_grid * 1000, SSR_grid, levels=levels, cmap='viridis_r', norm=LogNorm())
cbar = plt.colorbar(contour, ax=ax1, label='SSR (m²)')

# Add contour lines
contour_lines = ax1.contour(K_grid, R_grid * 1000, SSR_grid, levels=[0.1, 0.5, 1, 2, 5], colors='white', linewidths=0.5)
ax1.clabel(contour_lines, inline=True, fontsize=8, fmt='%.1f')

# Mark true values
ax1.scatter([K_true], [R_true * 1000], color='red', s=150, marker='*', 
            edgecolors='white', linewidths=1.5, zorder=5, label='True parameters')

# Draw line of constant R/K ratio (equifinality line)
K_line = np.linspace(5, 40, 100)
R_line = ratio_true * K_line
mask = (R_line >= R_range.min()) & (R_line <= R_range.max())
ax1.plot(K_line[mask], R_line[mask] * 1000, 'r--', linewidth=2, label=f'R/K = const.')

ax1.set_xlabel('K (m/day)')
ax1.set_ylabel('R (mm/day)')
ax1.set_title('Objective Function Surface')
ax1.legend(loc='upper left', fontsize=9)

# Right: Cross-section along equifinality line
ax2 = axes[1]

# Calculate SSR along the R/K = constant line
K_cross = np.linspace(5, 40, 50)
R_cross = ratio_true * K_cross
SSR_cross = [objective_function(k, r) for k, r in zip(K_cross, R_cross)]

ax2.plot(K_cross, SSR_cross, 'b-', linewidth=2)
ax2.axvline(K_true, color='red', linestyle='--', linewidth=1.5, label=f'True K = {K_true}')
ax2.set_xlabel('K (m/day)')
ax2.set_ylabel('SSR (m²)')
ax2.set_title('SSR Along Equifinality Line')
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)
ax2.set_ylim(0, 0.2)

plt.tight_layout()
plt.show()

print(f"\nAlong the red dashed line (constant R/K ratio), the SSR is nearly constant.")

> 💡 **Understanding the Plot**
> 
> - The **valley** in the objective function follows the line R/K = constant
> - Any parameter combination along this valley fits the data equally well
> - The optimizer can find many different "optimal" solutions depending on starting point
> - The right panel shows SSR is essentially flat along the equifinality line

## 4. Demonstrating Equifinality: Multiple Calibrations

Let's run gradient descent from different starting points and see where each ends up.

In [ ]:
def gradient_descent_2d(K_init, R_init, learning_rate_K=0.3, learning_rate_R=3e-7, 
                        max_iter=200, tol=1e-8):
    """
    Gradient descent for two parameters with momentum for smoother convergence.
    """
    K, R = K_init, R_init
    history = {'K': [K], 'R': [R], 'SSR': [objective_function(K, R)]}
    
    delta_K = 0.1
    delta_R = 1e-5
    
    # Momentum terms for smoother paths
    v_K, v_R = 0, 0
    momentum = 0.8
    
    for i in range(max_iter):
        # Calculate gradients using central differences
        grad_K = (objective_function(K + delta_K, R) - objective_function(K - delta_K, R)) / (2 * delta_K)
        grad_R = (objective_function(K, R + delta_R) - objective_function(K, R - delta_R)) / (2 * delta_R)
        
        # Update with momentum for smoother convergence
        v_K = momentum * v_K + learning_rate_K * grad_K
        v_R = momentum * v_R + learning_rate_R * grad_R
        
        K_new = K - v_K
        R_new = R - v_R
        
        # Keep parameters in bounds
        K_new = np.clip(K_new, 5, 40)
        R_new = np.clip(R_new, 0.0002, 0.003)
        
        # Store history (subsample to reduce clutter)
        history['K'].append(K_new)
        history['R'].append(R_new)
        history['SSR'].append(objective_function(K_new, R_new))
        
        # Check convergence
        if abs(K_new - K) < tol and abs(R_new - R) < tol * 1e-4:
            break
        
        K, R = K_new, R_new
    
    return history

In [ ]:
# Run calibration from different starting points
starting_points = [
    (8, 0.00055),    # Will converge to low K region
    (32, 0.0022),    # Will converge to high K region  
    (12, 0.0018),    # Off the line - high R
    (28, 0.0007),    # Off the line - low R
]

colors = ['#7B2CBF', '#FF6B35', '#2E8B8B', '#8B4513']  # purple, orange, teal, brown
results = []

fig, ax = plt.subplots(figsize=(7, 5))

# Plot objective function surface
levels = np.logspace(np.log10(SSR_grid.min() + 0.001), np.log10(SSR_grid.max()), 25)
contour = ax.contourf(K_grid, R_grid * 1000, SSR_grid, levels=levels, cmap='viridis_r', norm=LogNorm(), alpha=0.85)
plt.colorbar(contour, ax=ax, label='SSR (m²)')

# Draw equifinality line
ax.plot(K_line[mask], R_line[mask] * 1000, 'r--', linewidth=2.5, alpha=0.9, label='Equifinality line')

# Mark true parameters
ax.scatter([K_true], [R_true * 1000], color='red', s=200, marker='*', 
           edgecolors='white', linewidths=2, zorder=10, label='True parameters')

# Run and plot each calibration
print("Calibration results from different starting points:")
print("=" * 70)
print(f"{'Start':^18} {'Final K':^10} {'Final R':^12} {'SSR':^10} {'R/K':^12}")
print("-" * 70)

for idx, ((K_start, R_start), color) in enumerate(zip(starting_points, colors)):
    hist = gradient_descent_2d(K_start, R_start, learning_rate_K=0.15, learning_rate_R=1.5e-7, max_iter=300)
    results.append(hist)
    
    # Subsample path for cleaner visualization
    K_path = np.array(hist['K'][::5])
    R_path = np.array(hist['R'][::5]) * 1000
    
    # Plot optimization path
    ax.plot(K_path, R_path, '-', color=color, linewidth=2.5, alpha=0.9, zorder=3)
    
    # Mark start (square) and end (star)
    ax.scatter([hist['K'][0]], [hist['R'][0] * 1000], color=color, s=100, marker='s', 
               edgecolors='white', linewidths=1.5, zorder=6, label=f'Start {idx+1}')
    ax.scatter([hist['K'][-1]], [hist['R'][-1] * 1000], color=color, s=150, marker='*', 
               edgecolors='white', linewidths=1.5, zorder=6)
    
    # Print results
    K_final = hist['K'][-1]
    R_final = hist['R'][-1]
    SSR_final = hist['SSR'][-1]
    ratio_final = R_final / K_final
    print(f"K={K_start:4.0f}, R={R_start:.4f}  {K_final:8.2f}  {R_final*1000:8.4f} mm/d  {SSR_final:8.4f}  {ratio_final:.6f}")

ax.set_xlabel('Hydraulic Conductivity K (m/day)')
ax.set_ylabel('Recharge R (mm/day)')
ax.set_title('Optimization Paths from Different Starting Points')
ax.legend(loc='upper left', fontsize=9)
ax.set_xlim(5, 40)
ax.set_ylim(0.2, 3.0)

plt.tight_layout()
plt.show()

print("-" * 70)
print(f"True: K={K_true}, R={R_true*1000:.4f} mm/d, R/K={ratio_true:.6f}")

> ⚠️ **Critical Observation**
> 
> All calibration runs achieve similar (low) SSR values, but they find **very different parameter combinations**!
> 
> - The individual values of K and R are poorly constrained
> - But the **ratio R/K** is well constrained (similar across all runs)
> - This is **structural equifinality** - built into the model equations

## 5. Comparing Model Outputs

Despite different parameters, all calibrated models produce nearly identical outputs!

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

# Left: Head profiles
ax1.scatter(x_obs, h_obs, color='red', s=80, zorder=5, label='Observations', marker='o', edgecolors='darkred')

for i, (hist, color, (K_start, R_start)) in enumerate(zip(results, colors, starting_points)):
    K_cal = hist['K'][-1]
    R_cal = hist['R'][-1]
    x, h = simple_gw_model(K_cal, R_cal, L, h0, h1)
    ax1.plot(x, h, '-', color=color, linewidth=2, alpha=0.8, 
             label=f'K={K_cal:.1f}, R={R_cal*1000:.2f}')

ax1.set_xlabel('Distance (m)')
ax1.set_ylabel('Hydraulic Head (m)')
ax1.set_title('Head Profiles from Different Calibrations')
ax1.legend(loc='upper right', fontsize=8)
ax1.grid(True, alpha=0.3)
ax1.set_xlim(0, L)
ax1.set_ylim(94, 108)

# Right: Bar chart of parameters
x_pos = np.arange(len(results))
width = 0.35

K_values = [hist['K'][-1] for hist in results]
R_values = [hist['R'][-1] * 1000 for hist in results]

# Normalize for comparison
K_norm = [k / K_true for k in K_values]
R_norm = [r / (R_true * 1000) for r in R_values]

bars1 = ax2.bar(x_pos - width/2, K_norm, width, label='K / K_true', color='steelblue')
bars2 = ax2.bar(x_pos + width/2, R_norm, width, label='R / R_true', color='coral')

ax2.axhline(1, color='green', linestyle='--', linewidth=2, label='True value')
ax2.set_ylabel('Normalized Parameter Value')
ax2.set_xlabel('Calibration Run')
ax2.set_title('Parameter Estimates (Normalized)')
ax2.set_xticks(x_pos)
ax2.set_xticklabels([f'Run {i+1}' for i in range(len(results))])
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("Left: All models produce nearly identical head profiles!")
print("Right: But individual parameter values vary dramatically.")

## 6. Interactive Exploration of Equifinality

Use the sliders to explore how different K and R combinations along the equifinality line produce the same model output.

In [ ]:
# Create interactive figure
fig_int, (ax_model, ax_params) = plt.subplots(1, 2, figsize=(10, 4))

# Model panel
ax_model.scatter(x_obs, h_obs, color='red', s=80, zorder=5, label='Observations', marker='o', edgecolors='darkred')
model_line, = ax_model.plot([], [], 'b-', linewidth=2.5, label='Model')
ax_model.set_xlabel('Distance (m)')
ax_model.set_ylabel('Hydraulic Head (m)')
ax_model.set_title('Model vs Observations')
ax_model.set_xlim(0, L)
ax_model.set_ylim(94, 108)
ax_model.grid(True, alpha=0.3)
ax_model.legend(loc='upper right')

# Info text
info_text = ax_model.text(0.02, 0.98, '', transform=ax_model.transAxes, fontsize=10,
                          verticalalignment='top', fontfamily='monospace',
                          bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.9))

# Parameter space panel
contour_int = ax_params.contourf(K_grid, R_grid * 1000, SSR_grid, levels=levels, cmap='viridis_r', norm=LogNorm(), alpha=0.8)
ax_params.plot(K_line[mask], R_line[mask] * 1000, 'r--', linewidth=2, alpha=0.7, label='Equifinality line')
ax_params.scatter([K_true], [R_true * 1000], color='red', s=120, marker='*', edgecolors='white', zorder=5, label='True')
current_point, = ax_params.plot([], [], 'go', markersize=12, markeredgecolor='white', markeredgewidth=2)
ax_params.set_xlabel('K (m/day)')
ax_params.set_ylabel('R (mm/day)')
ax_params.set_title('Parameter Space')
ax_params.legend(loc='upper left', fontsize=9)

plt.tight_layout()

def update_interactive(K_value, R_value):
    """Update plots based on slider values."""
    R_actual = R_value / 1000  # Convert from mm/day to m/day
    
    # Calculate model
    x, h = simple_gw_model(K_value, R_actual, L, h0, h1)
    SSR = objective_function(K_value, R_actual)
    ratio = R_actual / K_value
    
    # Update model line
    model_line.set_data(x, h)
    
    # Update parameter marker
    current_point.set_data([K_value], [R_value])
    
    # Update info text
    info_text.set_text(f'K = {K_value:.1f} m/day\nR = {R_value:.2f} mm/day\nR/K = {ratio:.6f}\nSSR = {SSR:.4f} m²')
    
    fig_int.canvas.draw_idle()

# Create sliders
K_slider = widgets.FloatSlider(value=15.0, min=5.0, max=40.0, step=0.5,
                                description='K (m/day):', continuous_update=True,
                                style={'description_width': 'initial'},
                                layout=widgets.Layout(width='45%'))

R_slider = widgets.FloatSlider(value=1.0, min=0.2, max=3.0, step=0.02,
                                description='R (mm/day):', continuous_update=True,
                                style={'description_width': 'initial'},
                                layout=widgets.Layout(width='45%'))

# Link sliders to update function
out = widgets.interactive_output(update_interactive, {'K_value': K_slider, 'R_value': R_slider})

# Initialize plot
update_interactive(15.0, 1.0)

display(widgets.HBox([K_slider, R_slider]))
plt.show()

> 💡 **Try This**
> 
> 1. Set K=15, R=1.0 (true values) - note the SSR
> 2. Now try K=30, R=2.0 (double both) - SSR is almost identical!
> 3. Try K=7.5, R=0.5 (half both) - same SSR again!
> 
> The model fit stays the same as long as you maintain the R/K ratio.

## 7. Implications for Real-World Modeling

### What Causes Equifinality?

| Type | Cause | Example |
|------|-------|--------|
| **Structural** | Model equations contain parameter ratios/products | R/K ratio in this model |
| **Data-limited** | Insufficient observations to constrain parameters | Few wells, short time series |
| **Scale-related** | Effective parameters depend on measurement scale | Upscaled K vs. local K |

### Strategies to Address Equifinality

1. **Add independent observations**: Different data types constrain different parameters
   - Head observations constrain R/K ratio
   - Flow measurements (baseflow, spring discharge) constrain R directly
   - Tracer tests constrain K directly

2. **Use prior information**: Constrain parameters based on field measurements or literature

3. **Report uncertainty**: Use Monte Carlo or MCMC methods to explore the equifinal region

4. **Regularization**: Add penalties for unrealistic parameter values

## 8. Key Takeaways

> 📚 **Summary: Equifinality**
> 
> 1. **Equifinality** means multiple parameter combinations produce equally good model fits
> 
> 2. **Parameter correlations** create valleys in the objective function surface
> 
> 3. **Good fit ≠ correct parameters** - a well-calibrated model may have wrong parameter values
> 
> 4. **Ratios may be identifiable** even when individual parameters are not (R/K in our example)
> 
> 5. **Additional data types** help break parameter correlations

> ⚠️ **Practical Implications**
> 
> - Never trust a single "best" parameter set from calibration
> - Always explore parameter uncertainty
> - Consider what parameters your observations actually constrain
> - Be cautious when using calibrated parameters for predictions outside the calibration conditions

## References

- Beven, K., & Binley, A. (1992). The future of distributed models: Model calibration and uncertainty prediction. Hydrological Processes, 6(3), 279-298.
- Doherty, J., & Hunt, R. J. (2010). Approaches to highly parameterized inversion: A guide to using PEST for groundwater-model calibration. US Geological Survey Scientific Investigations Report 2010-5169.
- Hill, M. C., & Tiedeman, C. R. (2007). Effective Groundwater Model Calibration. Wiley.